In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
data_dir = project_dir / "Dataset" / "Dataset"
train_dir = data_dir / "Training"
general_dir = data_dir / "General Files"

people_path = train_dir / "la_mmh_training_person_dim.csv"
lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"
calendar_path = general_dir / "la_mmh_calendar_dim.csv"
lms_dim_path = general_dir / "la_mmh_lms_item_dim.csv"
video_dim_path = general_dir / "la_mmh_video_dim.csv"
pmi_path = train_dir / "la_mmh_training_person_module_instance_fact.csv"
module_path = general_dir / "la_mmh_module_instance_dim.csv"

for path in [people_path, lms_path, video_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")


In [ ]:
# Read timestamps from both activity sources in chunks and retain only needed columns.
def read_activity_times(path, source, chunksize=300_000):
    parts = []
    for chunk in pd.read_csv(path, usecols=["P_ID", "TIMESTAMP"], dtype={"P_ID": "string", "TIMESTAMP": "string"}, chunksize=chunksize):
        chunk["TIMESTAMP"] = pd.to_datetime(chunk["TIMESTAMP"], utc=True, errors="coerce")
        chunk = chunk.dropna(subset=["P_ID", "TIMESTAMP"])
        parts.append(chunk.rename(columns={"TIMESTAMP": "timestamp"}).assign(source=source))
    return pd.concat(parts, ignore_index=True)


In [ ]:
# Preparation complete. Run the analysis cell below for this monster type.

In [ ]:
events = pd.concat([read_activity_times(lms_path, "LMS"), read_activity_times(video_path, "Video")], ignore_index=True)
events = events.drop_duplicates(subset=["P_ID", "timestamp", "source"])
events = events.sort_values(["P_ID", "timestamp"])
events["gap_minutes"] = events.groupby("P_ID")["timestamp"].diff().dt.total_seconds() / 60
events["new_session"] = events["gap_minutes"].gt(15)
events["session_id"] = events["new_session"].groupby(events["P_ID"]).cumsum()
# Ignore the long gap that starts a new session when checking each session's internal gaps.
events["within_session_gap"] = events["gap_minutes"].where(~events["new_session"], 0)
sessions = events.groupby(["P_ID", "session_id"], as_index=False).agg(start=("timestamp", "min"), end=("timestamp", "max"), activity_rows=("timestamp", "size"), largest_gap_minutes=("within_session_gap", "max"))
sessions["duration_hours"] = (sessions["end"] - sessions["start"]).dt.total_seconds() / 3600
zombie_evidence = sessions.loc[(sessions["duration_hours"] > 3) & (sessions["largest_gap_minutes"].fillna(0) <= 15)].copy()
print(f"Found {zombie_evidence['P_ID'].nunique():,} candidate students")
display(zombie_evidence.head(20))

In [ ]:
zombie_submission = zombie_evidence[["P_ID"]].drop_duplicates().sort_values("P_ID").reset_index(drop=True)
zombie_submission.insert(0, "Monster Type", "Zombie")
display(zombie_submission)

In [ ]:
if not zombie_evidence.empty:
    display(zombie_evidence.head(1))
else:
    print("No activity sessions matched the Zombie rule.")

In [ ]:
print("Review note: Timestamps are converted to UTC. The activity tables are large, so this notebook can use substantial memory.")